# UEG — Universal Edge Gateway Classifier
## Training Notebook v3.0

Fixes from v2:
- Class-aware token filter — noise_gibberish and short classes no longer filtered out
- Early stopping — stops when f1_A plateaus, no wasted compute
- onnxscript added — ONNX export works
- Regex pre-classifier gate included as inference utility

### Setup
1. Add `HF_TOKEN` as Kaggle secret
2. Enable GPU
3. Run all cells

In [ ]:
# ============================================================
# CELL 1 — Install dependencies
# ============================================================
import subprocess
subprocess.run([
    'pip', 'install', '-q',
    'huggingface_hub', 'tokenizers', 'onnx',
    'onnxruntime', 'onnxscript',
    'scikit-learn', 'tqdm',
], check=True)
print('Done')

In [ ]:
# ============================================================
# CELL 2 — Imports and constants
# ============================================================
import os, json, math, re, time, random, shutil, tempfile
import numpy as np
from pathlib import Path
from typing import Optional
from dataclasses import dataclass, asdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.processors import TemplateProcessing

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download, HfApi
from kaggle_secrets import UserSecretsClient

WORK_DIR  = Path('/kaggle/working')
MODEL_DIR = WORK_DIR / 'ueg-model'
CKPT_DIR  = WORK_DIR / 'checkpoints'
CACHE_DIR = WORK_DIR / 'cache'
for d in [MODEL_DIR, CKPT_DIR, CACHE_DIR,
          MODEL_DIR/'weights', MODEL_DIR/'tokenizer',
          MODEL_DIR/'export',  MODEL_DIR/'labels']:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

secrets  = UserSecretsClient()
HF_TOKEN = secrets.get_secret('HF_TOKEN')
api      = HfApi(token=HF_TOKEN)
print('Ready')

In [ ]:
# ============================================================
# CELL 3 — Configuration
# ============================================================
@dataclass
class UEGConfig:
    # Architecture
    vocab_size:           int   = 32000
    max_seq_len:          int   = 128
    hidden_dim:           int   = 512
    num_layers:           int   = 6
    num_heads:            int   = 8
    ffn_dim:              int   = 2048
    dropout:              float = 0.1
    num_intent_classes:   int   = 22
    num_resource_classes: int   = 5
    # Training
    batch_size:           int   = 64
    learning_rate:        float = 3e-4
    weight_decay:         float = 0.01
    grad_clip:            float = 1.0
    warmup_ratio:         float = 0.05
    num_epochs:           int   = 20
    finetune_epochs:      int   = 3
    label_smoothing:      float = 0.1
    loss_weight_a:        float = 0.7
    loss_weight_b:        float = 0.3
    # Early stopping
    early_stop_patience:  int   = 4   # stop if no improvement for 4 epochs
    # Data
    global_min_tokens:    int   = 8   # for most classes
    short_min_tokens:     int   = 1   # for short-by-nature classes
    val_ratio:            float = 0.10
    test_ratio:           float = 0.05
    # Repos
    data_repo:            str   = 'rufatronics/ueg-training-data'
    model_repo:           str   = 'rufatronics/ueg-classifier'

CFG = UEGConfig()
with open(MODEL_DIR / 'config.json', 'w') as f:
    json.dump(asdict(CFG), f, indent=2)
print('Config saved')

In [ ]:
# ============================================================
# CELL 4 — Label maps
# ============================================================
INTENT_CLASSES = {
    1:'noise_gibberish',       2:'adversarial_probe',
    3:'greeting_open',         4:'phatic_social',
    5:'closure_gratitude',     6:'ui_command',
    7:'ambient_device_query',  8:'navigation_intent',
    9:'factoid_static',        10:'factoid_dynamic',
    11:'transactional_status', 12:'casual_open_chat',
    13:'code_task',            14:'data_structured',
    15:'document_structured',  16:'math_formal',
    17:'analysis_reasoning',   18:'long_form_creative',
    19:'domain_specialist',    20:'instruction_procedural',
    21:'debate_opinion',       22:'multilingual_task',
}
RESOURCE_CLASSES = {
    0:'hr_global', 1:'mr_regional', 2:'lr_emerging',
    3:'mul_mix',   4:'noise_nonlinguistic',
}
INTENT_LABEL2ID   = {v: i for i, v in enumerate(INTENT_CLASSES.values())}
INTENT_ID2LABEL   = {i: v for i, v in enumerate(INTENT_CLASSES.values())}
RESOURCE_LABEL2ID = {v: i for i, v in RESOURCE_CLASSES.items()}
RESOURCE_ID2LABEL = {i: v for i, v in RESOURCE_CLASSES.items()}

# Classes where short text is expected and correct — DO NOT filter these
# 0-indexed intent label ids
SHORT_OK_CLASSES = {
    INTENT_LABEL2ID['noise_gibberish'],    # keyboard mash, symbols — always short
    INTENT_LABEL2ID['greeting_open'],      # 'Hi', 'Hey', 'Hello'
    INTENT_LABEL2ID['phatic_social'],      # 'How are you'
    INTENT_LABEL2ID['closure_gratitude'],  # 'Thanks', 'Bye'
    INTENT_LABEL2ID['ui_command'],         # 'Dark mode', 'Logout'
    INTENT_LABEL2ID['ambient_device_query'], # 'What time is it'
    INTENT_LABEL2ID['navigation_intent'],  # 'Go to settings'
}
print(f'Short-ok classes (0-indexed): {SHORT_OK_CLASSES}')

with open(MODEL_DIR/'labels'/'intent_classes.json', 'w') as f:
    json.dump({'label2id': INTENT_LABEL2ID, 'id2label': INTENT_ID2LABEL}, f, indent=2)
with open(MODEL_DIR/'labels'/'resource_classes.json', 'w') as f:
    json.dump({'label2id': RESOURCE_LABEL2ID, 'id2label': RESOURCE_ID2LABEL}, f, indent=2)
print(f'Intent: {len(INTENT_LABEL2ID)} | Resource: {len(RESOURCE_LABEL2ID)}')

In [ ]:
# ============================================================
# CELL 5 — Checkpoint Manager
# ============================================================
class CheckpointManager:
    STATE_FILE = 'training_state.json'
    CKPT_FILE  = 'checkpoint_latest.pt'
    BEST_FILE  = 'checkpoint_best.pt'

    def __init__(self, model_repo, local_dir, hf_token):
        self.repo      = model_repo
        self.local_dir = local_dir
        self.token     = hf_token
        self.api       = HfApi(token=hf_token)
        self._ensure_repo()

    def _ensure_repo(self):
        try:
            self.api.model_info(self.repo, token=self.token)
            print(f'[CKPT] Repo exists: {self.repo}')
        except Exception:
            self.api.create_repo(self.repo, repo_type='model', private=False, token=self.token)
            print(f'[CKPT] Repo created: {self.repo}')

    def load_state(self):
        try:
            path = hf_hub_download(
                repo_id=self.repo, filename=self.STATE_FILE,
                repo_type='model', token=self.token, force_download=True,
            )
            with open(path) as f: state = json.load(f)
            print(f'[CKPT] Found checkpoint — epoch {state["epoch"]} | best_f1={state["best_f1"]:.4f}')
            return state
        except Exception:
            print('[CKPT] No checkpoint — fresh start')
            return None

    def load_weights(self, model, optimizer=None, scheduler=None, filename=None):
        fname = filename or self.CKPT_FILE
        try:
            path = hf_hub_download(
                repo_id=self.repo, filename=fname,
                repo_type='model', token=self.token, force_download=True,
            )
            ckpt = torch.load(path, map_location=DEVICE)
            model.load_state_dict(ckpt['model'])
            if optimizer and 'optimizer' in ckpt: optimizer.load_state_dict(ckpt['optimizer'])
            if scheduler and 'scheduler' in ckpt: scheduler.load_state_dict(ckpt['scheduler'])
            print(f'[CKPT] Loaded {fname}')
            return True
        except Exception as e:
            print(f'[CKPT] Load failed: {e}')
            return False

    def save(self, model, optimizer, scheduler, epoch, total_epochs,
             best_f1, history, phase, is_best=False):
        tmp = tempfile.mkdtemp()
        try:
            ckpt = {
                'model':     model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'scheduler': scheduler.state_dict(),
                'epoch': epoch, 'phase': phase,
            }
            torch.save(ckpt, os.path.join(tmp, self.CKPT_FILE))
            if is_best:
                torch.save(ckpt, os.path.join(tmp, self.BEST_FILE))

            state = {
                'epoch': epoch, 'total_epochs': total_epochs,
                'phase': phase, 'best_f1': best_f1,
                'history': history,
                'saved_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
            }
            with open(os.path.join(tmp, self.STATE_FILE), 'w') as f:
                json.dump(state, f, indent=2)

            files = [self.CKPT_FILE, self.STATE_FILE]
            if is_best: files.append(self.BEST_FILE)

            for fname in files:
                fpath = os.path.join(tmp, fname)
                if os.path.exists(fpath):
                    self.api.upload_file(
                        path_or_fileobj=fpath,
                        path_in_repo=fname,
                        repo_id=self.repo, repo_type='model',
                        commit_message=f'Epoch {epoch}/{total_epochs} | {phase} | f1={best_f1:.4f}',
                    )
            star = ' ★ NEW BEST' if is_best else ''
            print(f'[CKPT] Saved epoch {epoch}/{total_epochs} | {phase} | f1={best_f1:.4f}{star}')
        except Exception as e:
            print(f'[CKPT] Save failed: {e} — training continues')
        finally:
            shutil.rmtree(tmp, ignore_errors=True)

ckpt_manager = CheckpointManager(CFG.model_repo, CKPT_DIR, HF_TOKEN)
print('Checkpoint manager ready')

In [ ]:
# ============================================================
# CELL 6 — Load dataset
# ============================================================
print(f'Loading from {CFG.data_repo}...')
all_examples = []
skipped = 0

for class_id, label in INTENT_CLASSES.items():
    filename = f'data/class_{class_id:02d}_{label}.jsonl'
    try:
        local = hf_hub_download(
            repo_id=CFG.data_repo, filename=filename,
            repo_type='dataset', token=HF_TOKEN,
            cache_dir=str(CACHE_DIR), force_download=False,
        )
        count = 0
        with open(local, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line: continue
                try:
                    ex  = json.loads(line)
                    txt = ex.get('text','').strip()
                    if not txt: skipped += 1; continue
                    il  = ex.get('intent_class_label','')
                    rl  = ex.get('resource_class','hr_global')
                    if il not in INTENT_LABEL2ID: skipped += 1; continue
                    if rl not in RESOURCE_LABEL2ID: rl = 'hr_global'
                    all_examples.append({
                        'text':           txt,
                        'intent_label':   INTENT_LABEL2ID[il],
                        'resource_label': RESOURCE_LABEL2ID[rl],
                    })
                    count += 1
                except Exception: skipped += 1
        print(f'  [{class_id:2d}] {label:<28} {count:,}')
    except Exception as e:
        print(f'  [{class_id:2d}] {label:<28} FAILED: {e}')

print(f'\nTotal: {len(all_examples):,} | Skipped: {skipped:,}')

In [ ]:
# ============================================================
# CELL 7 — Class-aware filter + stratified split
# Short classes (gibberish, greetings etc) keep min 1 token
# Everything else keeps min 8 tokens
# ============================================================
before = len(all_examples)
filtered = []
for ex in all_examples:
    tokens = len(ex['text'].split())
    min_t  = CFG.short_min_tokens if ex['intent_label'] in SHORT_OK_CLASSES \
             else CFG.global_min_tokens
    if tokens >= min_t:
        filtered.append(ex)

all_examples = filtered
removed = before - len(all_examples)
print(f'After class-aware filter: {len(all_examples):,} (removed {removed:,})')

# Show per-class counts after filtering
from collections import Counter
counts = Counter(ex['intent_label'] for ex in all_examples)
for i, label in INTENT_ID2LABEL.items():
    print(f'  {i:2d} {label:<28} {counts.get(i,0):,}')

# Stratified split
labels_all = [ex['intent_label'] for ex in all_examples]
train_val, test_data = train_test_split(
    all_examples, test_size=CFG.test_ratio,
    stratify=labels_all, random_state=SEED,
)
train_data, val_data = train_test_split(
    train_val,
    test_size=CFG.val_ratio / (1 - CFG.test_ratio),
    stratify=[ex['intent_label'] for ex in train_val],
    random_state=SEED,
)
print(f'\nTrain: {len(train_data):,} | Val: {len(val_data):,} | Test: {len(test_data):,}')

In [ ]:
# ============================================================
# CELL 8 — Tokenizer (load from HF if exists, else train)
# ============================================================
tok_dir  = MODEL_DIR / 'tokenizer'
tok_file = tok_dir / 'tokenizer.json'
tok_cfg  = tok_dir / 'tokenizer_config.json'

tokenizer_loaded = False
try:
    p  = hf_hub_download(repo_id=CFG.model_repo, filename='tokenizer/tokenizer.json',
                          repo_type='model', token=HF_TOKEN, force_download=True)
    pc = hf_hub_download(repo_id=CFG.model_repo, filename='tokenizer/tokenizer_config.json',
                          repo_type='model', token=HF_TOKEN, force_download=True)
    shutil.copy(p, str(tok_file)); shutil.copy(pc, str(tok_cfg))
    tokenizer = Tokenizer.from_file(str(tok_file))
    with open(tok_cfg) as f: tc = json.load(f)
    pad_id = tc['pad_id']; cls_id = tc['cls_id']; sep_id = tc['sep_id']
    tokenizer_loaded = True
    print(f'Tokenizer loaded from HF — vocab: {tokenizer.get_vocab_size():,}')
except Exception:
    print('Training tokenizer fresh...')

if not tokenizer_loaded:
    corpus_path = str(CACHE_DIR / 'corpus.txt')
    with open(corpus_path, 'w', encoding='utf-8') as f:
        for ex in train_data: f.write(ex['text'] + '\n')

    tokenizer = Tokenizer(BPE(unk_token='[UNK]'))
    tokenizer.pre_tokenizer = ByteLevel()
    trainer = BpeTrainer(
        vocab_size=CFG.vocab_size,
        special_tokens=['[PAD]','[UNK]','[CLS]','[SEP]','[MASK]'],
        min_frequency=2, show_progress=True,
    )
    tokenizer.train([corpus_path], trainer)
    pad_id = tokenizer.token_to_id('[PAD]')
    cls_id = tokenizer.token_to_id('[CLS]')
    sep_id = tokenizer.token_to_id('[SEP]')
    tokenizer.post_processor = TemplateProcessing(
        single='[CLS] $A [SEP]',
        special_tokens=[('[CLS]', cls_id), ('[SEP]', sep_id)],
    )
    tokenizer.enable_padding(pad_id=pad_id, pad_token='[PAD]', length=CFG.max_seq_len)
    tokenizer.enable_truncation(max_length=CFG.max_seq_len)
    tokenizer.save(str(tok_file))
    tc = {'vocab_size': CFG.vocab_size, 'max_seq_len': CFG.max_seq_len,
          'pad_id': pad_id, 'cls_id': cls_id, 'sep_id': sep_id,
          'pad_token':'[PAD]','cls_token':'[CLS]','sep_token':'[SEP]'}
    with open(tok_cfg, 'w') as f: json.dump(tc, f, indent=2)
    api.upload_folder(
        folder_path=str(tok_dir), path_in_repo='tokenizer',
        repo_id=CFG.model_repo, repo_type='model',
        commit_message='Add tokenizer',
    )
    print(f'Tokenizer trained and pushed — vocab: {tokenizer.get_vocab_size():,}')

In [ ]:
# ============================================================
# CELL 9 — Dataset and DataLoader
# ============================================================
class UEGDataset(Dataset):
    def __init__(self, examples, tokenizer, max_len):
        self.examples  = examples
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self): return len(self.examples)

    def __getitem__(self, idx):
        ex  = self.examples[idx]
        enc = self.tokenizer.encode(ex['text'])
        ids = enc.ids[:self.max_len]
        msk = enc.attention_mask[:self.max_len]
        pad = self.max_len - len(ids)
        ids += [pad_id] * pad; msk += [0] * pad
        return {
            'input_ids':      torch.tensor(ids, dtype=torch.long),
            'attention_mask': torch.tensor(msk, dtype=torch.long),
            'intent_label':   torch.tensor(ex['intent_label'],   dtype=torch.long),
            'resource_label': torch.tensor(ex['resource_label'], dtype=torch.long),
        }

train_loader = DataLoader(UEGDataset(train_data, tokenizer, CFG.max_seq_len),
    batch_size=CFG.batch_size, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(UEGDataset(val_data,   tokenizer, CFG.max_seq_len),
    batch_size=CFG.batch_size*2, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(UEGDataset(test_data,  tokenizer, CFG.max_seq_len),
    batch_size=CFG.batch_size*2, shuffle=False, num_workers=0, pin_memory=True)

print(f'Train: {len(train_loader):,} batches | Val: {len(val_loader):,} | Test: {len(test_loader):,}')
# Note: num_workers=0 eliminates the AssertionError multiprocessing noise

In [ ]:
# ============================================================
# CELL 10 — Model architecture
# ============================================================
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, hidden_dim, num_heads, dropout):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = hidden_dim // num_heads
        self.scale     = self.head_dim ** -0.5
        self.qkv  = nn.Linear(hidden_dim, hidden_dim * 3, bias=False)
        self.proj = nn.Linear(hidden_dim, hidden_dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        B, T, C = x.shape
        qkv = self.qkv(x).reshape(B,T,3,self.num_heads,self.head_dim).permute(2,0,3,1,4)
        q, k, v = qkv.unbind(0)
        attn = (q @ k.transpose(-2,-1)) * self.scale
        if mask is not None:
            attn = attn.masked_fill(mask.unsqueeze(1).unsqueeze(2)==0, float('-inf'))
        attn = self.drop(F.softmax(attn, dim=-1))
        return self.proj((attn @ v).transpose(1,2).reshape(B,T,C))

class FeedForward(nn.Module):
    def __init__(self, hidden_dim, ffn_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, ffn_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ffn_dim, hidden_dim), nn.Dropout(dropout),
        )
    def forward(self, x): return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, hidden_dim, num_heads, ffn_dim, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.attn  = MultiHeadSelfAttention(hidden_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.ffn   = FeedForward(hidden_dim, ffn_dim, dropout)
    def forward(self, x, mask=None):
        x = x + self.attn(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x

class UEGModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.token_emb    = nn.Embedding(cfg.vocab_size, cfg.hidden_dim, padding_idx=pad_id)
        self.pos_emb      = nn.Embedding(cfg.max_seq_len, cfg.hidden_dim)
        self.emb_drop     = nn.Dropout(cfg.dropout)
        self.emb_norm     = nn.LayerNorm(cfg.hidden_dim)
        self.encoder      = nn.ModuleList([
            TransformerBlock(cfg.hidden_dim, cfg.num_heads, cfg.ffn_dim, cfg.dropout)
            for _ in range(cfg.num_layers)
        ])
        self.encoder_norm = nn.LayerNorm(cfg.hidden_dim)
        self.head_a = nn.Sequential(
            nn.Linear(cfg.hidden_dim, cfg.hidden_dim), nn.GELU(),
            nn.Dropout(cfg.dropout), nn.Linear(cfg.hidden_dim, cfg.num_intent_classes),
        )
        self.head_b = nn.Sequential(
            nn.Linear(cfg.hidden_dim, cfg.hidden_dim//2), nn.GELU(),
            nn.Dropout(cfg.dropout), nn.Linear(cfg.hidden_dim//2, cfg.num_resource_classes),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding): nn.init.trunc_normal_(m.weight, std=0.02)
            elif isinstance(m, nn.LayerNorm): nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, input_ids, attention_mask):
        B, T = input_ids.shape
        pos  = torch.arange(T, device=input_ids.device).unsqueeze(0)
        x    = self.emb_norm(self.emb_drop(self.token_emb(input_ids) + self.pos_emb(pos)))
        for block in self.encoder: x = block(x, attention_mask)
        cls = self.encoder_norm(x)[:, 0, :]
        return self.head_a(cls), self.head_b(cls)

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def freeze_encoder(self):
        for p in list(self.token_emb.parameters()) + list(self.pos_emb.parameters()) + \
                 list(self.encoder.parameters()) + list(self.encoder_norm.parameters()):
            p.requires_grad = False
        print('Encoder frozen')

    def unfreeze_encoder(self):
        for p in self.parameters(): p.requires_grad = True
        print('Encoder unfrozen')

model  = UEGModel(CFG).to(DEVICE)
params = model.count_parameters()
print(f'Parameters: {params:,} ({params/1e6:.1f}M)')

In [ ]:
# ============================================================
# CELL 11 — Loss, optimizer, scheduler
# ============================================================
class UEGLoss(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.wa  = cfg.loss_weight_a; self.wb = cfg.loss_weight_b
        self.ce_a = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
        self.ce_b = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    def forward(self, la, lb, ya, yb):
        loss_a = self.ce_a(la, ya); loss_b = self.ce_b(lb, yb)
        return self.wa * loss_a + self.wb * loss_b, loss_a, loss_b

criterion    = UEGLoss(CFG)
total_steps  = len(train_loader) * CFG.num_epochs
warmup_steps = int(total_steps * CFG.warmup_ratio)

optimizer = AdamW(model.parameters(), lr=CFG.learning_rate,
                  weight_decay=CFG.weight_decay, betas=(0.9,0.999), eps=1e-8)

def get_lr(step):
    if step < warmup_steps: return step / max(1, warmup_steps)
    p = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return max(0.05, 0.5 * (1 + math.cos(math.pi * p)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, get_lr)
print(f'Steps: {total_steps:,} | Warmup: {warmup_steps:,} | Early stop patience: {CFG.early_stop_patience}')

In [ ]:
# ============================================================
# CELL 12 — Resume detection
# ============================================================
existing_state = ckpt_manager.load_state()

start_epoch = 1
best_f1     = 0.0
history     = []
phase       = 'main'
step        = 0
no_improve  = 0  # early stopping counter

if existing_state:
    start_epoch = existing_state['epoch'] + 1
    best_f1     = existing_state['best_f1']
    history     = existing_state['history']
    phase       = existing_state['phase']
    # Recalculate no_improve from history
    main_history = [h for h in history if h['phase'] == 'main']
    if main_history:
        for h in reversed(main_history):
            if h['val_f1_a'] >= best_f1: break
            no_improve += 1

    ok = ckpt_manager.load_weights(model, optimizer, scheduler)
    if ok:
        step = (start_epoch - 1) * len(train_loader)
        print(f'Resumed epoch {start_epoch-1} | phase={phase} | best_f1={best_f1:.4f} | no_improve={no_improve}')
        if phase == 'finetune': model.freeze_encoder()
    else:
        print('Weight load failed — starting fresh')
        start_epoch = 1; existing_state = None
else:
    print(f'Fresh start — max {CFG.num_epochs} epochs, patience={CFG.early_stop_patience}')

In [ ]:
# ============================================================
# CELL 13 — Train + eval functions
# ============================================================
def train_epoch(model, loader, optimizer, scheduler, criterion, step):
    model.train()
    total_loss = correct_a = correct_b = total = 0
    pbar = tqdm(loader, desc='Train', leave=False)
    for batch in pbar:
        ids = batch['input_ids'].to(DEVICE)
        msk = batch['attention_mask'].to(DEVICE)
        ya  = batch['intent_label'].to(DEVICE)
        yb  = batch['resource_label'].to(DEVICE)
        optimizer.zero_grad()
        la, lb = model(ids, msk)
        loss, _, _ = criterion(la, lb, ya, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
        optimizer.step(); scheduler.step(); step += 1
        bs = ids.size(0)
        total_loss += loss.item() * bs
        correct_a  += (la.argmax(-1)==ya).sum().item()
        correct_b  += (lb.argmax(-1)==yb).sum().item()
        total      += bs
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc_a': f'{correct_a/total:.3f}'})
    return total_loss/total, correct_a/total, correct_b/total, step

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    pa, ta, pb, tb = [], [], [], []
    for batch in tqdm(loader, desc='Eval', leave=False):
        ids = batch['input_ids'].to(DEVICE)
        msk = batch['attention_mask'].to(DEVICE)
        ya  = batch['intent_label'].to(DEVICE)
        yb  = batch['resource_label'].to(DEVICE)
        la, lb = model(ids, msk)
        loss, _, _ = criterion(la, lb, ya, yb)
        total_loss += loss.item() * ids.size(0)
        pa.extend(la.argmax(-1).cpu().tolist())
        ta.extend(ya.cpu().tolist())
        pb.extend(lb.argmax(-1).cpu().tolist())
        tb.extend(yb.cpu().tolist())
    n = len(pa)
    return {
        'loss':     total_loss/n,
        'acc_a':    sum(p==t for p,t in zip(pa,ta))/n,
        'f1_a':     f1_score(ta, pa, average='macro'),
        'acc_b':    sum(p==t for p,t in zip(pb,tb))/n,
        'f1_b':     f1_score(tb, pb, average='macro'),
        'preds_a':  pa, 'truths_a': ta,
    }

print('Functions ready')

In [ ]:
# ============================================================
# CELL 14 — Main training with early stopping
# ============================================================
if phase == 'main' and start_epoch <= CFG.num_epochs:
    print(f'Main training: epochs {start_epoch} → {CFG.num_epochs} (patience={CFG.early_stop_patience})')
    print('=' * 62)

    for epoch in range(start_epoch, CFG.num_epochs + 1):
        t0 = time.time()
        train_loss, train_acc_a, _, step = train_epoch(
            model, train_loader, optimizer, scheduler, criterion, step
        )
        val_m   = evaluate(model, val_loader, criterion)
        elapsed = time.time() - t0

        is_best = val_m['f1_a'] > best_f1
        if is_best:
            best_f1    = val_m['f1_a']
            no_improve = 0
        else:
            no_improve += 1

        record = {
            'epoch': epoch, 'phase': 'main',
            'train_loss': round(train_loss, 4),
            'val_loss':   round(val_m['loss'], 4),
            'val_acc_a':  round(val_m['acc_a'], 4),
            'val_f1_a':   round(val_m['f1_a'],  4),
            'val_acc_b':  round(val_m['acc_b'],  4),
            'val_f1_b':   round(val_m['f1_b'],   4),
            'elapsed':    round(elapsed, 1),
        }
        history.append(record)

        stop_note = f' [patience {no_improve}/{CFG.early_stop_patience}]' if not is_best else ''
        print(
            f"Epoch {epoch:2d}/{CFG.num_epochs} | "
            f"loss {train_loss:.4f}→{val_m['loss']:.4f} | "
            f"f1_A {val_m['f1_a']:.4f} | "
            f"f1_B {val_m['f1_b']:.4f} | "
            f"{elapsed:.0f}s"
            + (' ★' if is_best else stop_note)
        )

        ckpt_manager.save(
            model, optimizer, scheduler,
            epoch=epoch, total_epochs=CFG.num_epochs,
            best_f1=best_f1, history=history,
            phase='main', is_best=is_best,
        )

        # Early stopping
        if no_improve >= CFG.early_stop_patience:
            print(f'\nEarly stopping — no improvement for {CFG.early_stop_patience} epochs')
            print(f'Best was epoch {epoch - no_improve} with f1_A={best_f1:.4f}')
            break

    print(f'\nMain training done — best f1_A: {best_f1:.4f}')
    phase = 'finetune'
else:
    print('Skipping main training')

In [ ]:
# ============================================================
# CELL 15 — Finetune heads
# ============================================================
ft_start = 1
if phase == 'finetune':
    print('Loading best checkpoint for head fine-tuning...')
    ckpt_manager.load_weights(model, None, None, filename='checkpoint_best.pt')
    model.freeze_encoder()

    head_optimizer = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=CFG.learning_rate * 0.1, weight_decay=CFG.weight_decay,
    )
    ft_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        head_optimizer,
        T_max=len(train_loader) * CFG.finetune_epochs, eta_min=1e-6,
    )

    if existing_state and existing_state.get('phase') == 'finetune':
        ft_start = existing_state['epoch'] + 1
        print(f'Resuming finetune from epoch {ft_start}')

    ft_no_improve = 0
    step = 0
    print(f'Fine-tuning: {ft_start} → {CFG.finetune_epochs} epochs')
    print('=' * 62)

    for epoch in range(ft_start, CFG.finetune_epochs + 1):
        t0 = time.time()
        train_loss, _, _, step = train_epoch(
            model, train_loader, head_optimizer, ft_scheduler, criterion, step
        )
        val_m   = evaluate(model, val_loader, criterion)
        elapsed = time.time() - t0

        is_best = val_m['f1_a'] > best_f1
        if is_best:
            best_f1      = val_m['f1_a']
            ft_no_improve = 0
        else:
            ft_no_improve += 1

        history.append({
            'epoch': epoch, 'phase': 'finetune',
            'train_loss': round(train_loss, 4),
            'val_f1_a':   round(val_m['f1_a'], 4),
            'val_f1_b':   round(val_m['f1_b'], 4),
            'elapsed':    round(elapsed, 1),
        })

        print(
            f"FT {epoch}/{CFG.finetune_epochs} | "
            f"f1_A {val_m['f1_a']:.4f} | "
            f"f1_B {val_m['f1_b']:.4f} | "
            f"{elapsed:.0f}s"
            + (' ★' if is_best else f' [no improve {ft_no_improve}]')
        )

        ckpt_manager.save(
            model, head_optimizer, ft_scheduler,
            epoch=epoch, total_epochs=CFG.finetune_epochs,
            best_f1=best_f1, history=history,
            phase='finetune', is_best=is_best,
        )

        if ft_no_improve >= 2:  # tighter patience for finetune
            print(f'Finetune early stop — no improvement for 2 epochs')
            break

    print(f'\nFinetune done — best f1_A: {best_f1:.4f}')

In [ ]:
# ============================================================
# CELL 16 — Final test evaluation
# ============================================================
print('Loading best weights...')
ckpt_manager.load_weights(model, None, None, filename='checkpoint_best.pt')
model.unfreeze_encoder()

test_m = evaluate(model, test_loader, criterion)

print('\n' + '='*62)
print('FINAL TEST RESULTS')
print('='*62)
print(f"  Head A (Intent)   — Acc: {test_m['acc_a']:.4f} | F1: {test_m['f1_a']:.4f}")
print(f"  Head B (Resource) — Acc: {test_m['acc_b']:.4f} | F1: {test_m['f1_b']:.4f}")
print(f"  Loss              : {test_m['loss']:.4f}")
print('='*62)

print('\nPer-class (Head A):')
print(classification_report(
    test_m['truths_a'], test_m['preds_a'],
    target_names=list(INTENT_LABEL2ID.keys()), digits=4,
))

torch.save(model.state_dict(), MODEL_DIR/'weights'/'ueg_final.pt')
with open(MODEL_DIR/'test_results.json','w') as f:
    json.dump({k:v for k,v in test_m.items() if k not in ('preds_a','truths_a')}, f, indent=2)
with open(MODEL_DIR/'training_history.json','w') as f:
    json.dump(history, f, indent=2)
print('Saved')

In [ ]:
# ============================================================
# CELL 17 — ONNX export + benchmark
# ============================================================
import onnx
import onnxruntime as ort

model.eval()
onnx_path  = str(MODEL_DIR/'export'/'ueg_model.onnx')
dummy_ids  = torch.zeros(1, CFG.max_seq_len, dtype=torch.long).to(DEVICE)
dummy_mask = torch.ones(1,  CFG.max_seq_len, dtype=torch.long).to(DEVICE)

with torch.no_grad():
    torch.onnx.export(
        model, (dummy_ids, dummy_mask), onnx_path,
        input_names=['input_ids','attention_mask'],
        output_names=['logits_intent','logits_resource'],
        dynamic_axes={
            'input_ids':       {0:'batch'},
            'attention_mask':  {0:'batch'},
            'logits_intent':   {0:'batch'},
            'logits_resource': {0:'batch'},
        },
        opset_version=17, do_constant_folding=True,
    )

onnx.checker.check_model(onnx.load(onnx_path))
print('ONNX verified')

sess      = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
np_ids    = np.zeros((1, CFG.max_seq_len), dtype=np.int64)
np_mask   = np.ones((1,  CFG.max_seq_len), dtype=np.int64)

for _ in range(10): sess.run(None, {'input_ids': np_ids, 'attention_mask': np_mask})

times = []
for _ in range(200):
    t = time.perf_counter()
    sess.run(None, {'input_ids': np_ids, 'attention_mask': np_mask})
    times.append((time.perf_counter() - t) * 1000)

p50 = np.percentile(times, 50)
p95 = np.percentile(times, 95)
p99 = np.percentile(times, 99)
print(f'Latency — p50: {p50:.2f}ms | p95: {p95:.2f}ms | p99: {p99:.2f}ms')
print(f'Spec <5ms: {"✓ PASS" if p95 < 5.0 else "✗ FAIL — applying quantization"}')

if p95 >= 5.0:
    from onnxruntime.quantization import quantize_dynamic, QuantType
    q_path = str(MODEL_DIR/'export'/'ueg_model_int8.onnx')
    quantize_dynamic(onnx_path, q_path, weight_type=QuantType.QInt8)
    sess_q  = ort.InferenceSession(q_path, providers=['CPUExecutionProvider'])
    times_q = []
    for _ in range(200):
        t = time.perf_counter()
        sess_q.run(None, {'input_ids': np_ids, 'attention_mask': np_mask})
        times_q.append((time.perf_counter() - t) * 1000)
    p95_q = np.percentile(times_q, 95)
    print(f'INT8 p95: {p95_q:.2f}ms | {"✓ PASS" if p95_q < 5.0 else "✗ Still above"}')

with open(MODEL_DIR/'export'/'benchmark.json','w') as f:
    json.dump({'p50_ms': p50, 'p95_ms': p95, 'p99_ms': p99}, f, indent=2)

In [ ]:
# ============================================================
# CELL 18 — Regex pre-classifier gate
# Saved as inference_gate.py alongside the model
# ============================================================
gate_code = '''
import re

# Intent class IDs (1-indexed, matching taxonomy)
CLASS_NOISE_GIBBERISH = 1

def preprocess_input(text: str):
    """
    Pre-classifier regex gate.
    Catches noise_gibberish before the model sees it.
    Returns (text, forced_class_id) or (text, None) to proceed to model.
    forced_class_id is 1-indexed to match the taxonomy.
    """
    stripped = text.strip()

    # Empty input
    if len(stripped) == 0:
        return stripped, CLASS_NOISE_GIBBERISH

    # Pure whitespace or single char
    if len(stripped) < 2:
        return stripped, CLASS_NOISE_GIBBERISH

    # Pure non-alphanumeric across all scripts
    # Keeps Arabic, Hindi, Chinese, Latin alphanumeric
    alnum_pattern = re.compile(
        r"[a-zA-Z0-9"
        r"\u0600-\u06FF"   # Arabic
        r"\u0900-\u097F"   # Hindi/Devanagari
        r"\u4e00-\u9fff"   # Chinese
        r"\uAC00-\uD7AF"   # Korean
        r"\u0400-\u04FF"   # Cyrillic
        r"\u0370-\u03FF"   # Greek
        r"]"
    )
    if not alnum_pattern.search(stripped):
        return stripped, CLASS_NOISE_GIBBERISH

    # Repetitive single character — 'zzzzz', 'aaaaa', '11111'
    if re.fullmatch(r"(.)\\1{4,}", stripped):
        return stripped, CLASS_NOISE_GIBBERISH

    # Keyboard mash detection — Latin only, no vowels in long sequences
    latin_only = re.sub(r"[^a-zA-Z]", "", stripped.lower())
    if len(latin_only) > 6:
        vowel_ratio = sum(1 for c in latin_only if c in "aeiou") / len(latin_only)
        if vowel_ratio < 0.05:
            return stripped, CLASS_NOISE_GIBBERISH

    # Random alternating keys — 'asdfasdf', 'qwerty'
    keyboard_rows = [
        set("qwertyuiop"), set("asdfghjkl"), set("zxcvbnm")
    ]
    if len(latin_only) >= 6:
        unique_chars = set(latin_only)
        for row in keyboard_rows:
            if unique_chars.issubset(row) and len(unique_chars) >= 4:
                return stripped, CLASS_NOISE_GIBBERISH

    # All good — pass to model
    return stripped, None


def classify_with_gate(text: str, model_fn) -> dict:
    """
    Full inference pipeline.
    model_fn: callable that takes text and returns (intent_id, resource_id, confidences)
    """
    text, forced = preprocess_input(text)

    if forced is not None:
        return {
            "intent_class_id":    forced,
            "intent_class_label": "noise_gibberish",
            "resource_class":     "noise_nonlinguistic",
            "confidence":         1.0,
            "source":             "regex_gate",
        }

    return model_fn(text)
'''

gate_path = MODEL_DIR / 'export' / 'inference_gate.py'
with open(gate_path, 'w') as f:
    f.write(gate_code)
print(f'Regex gate saved to {gate_path}')

In [ ]:
# ============================================================
# CELL 19 — Upload everything to HuggingFace
# ============================================================
print('Uploading final model...')
api.upload_folder(
    folder_path=str(MODEL_DIR),
    repo_id=CFG.model_repo,
    repo_type='model',
    commit_message=f'UEG v2.0 — f1_A={best_f1:.4f} | {params/1e6:.1f}M params | all fixes applied',
)
print(f'Done — https://huggingface.co/{CFG.model_repo}')

In [ ]:
# ============================================================
# CELL 20 — Summary
# ============================================================
print('\n' + '='*62)
print('  UEG v2.0 — TRAINING COMPLETE')
print('='*62)
print(f'  Parameters  : {params/1e6:.1f}M')
print(f'  Best f1_A   : {best_f1:.4f}')
print(f'  Test f1_A   : {test_m["f1_a"]:.4f}')
print(f'  Test f1_B   : {test_m["f1_b"]:.4f}')
print(f'  ONNX p50    : {p50:.2f}ms')
print(f'  ONNX p95    : {p95:.2f}ms')
print(f'  Epochs run  : {len([h for h in history if h["phase"]=="main"])} main + {len([h for h in history if h["phase"]=="finetune"])} finetune')
print(f'  Model       : https://huggingface.co/{CFG.model_repo}')
print('='*62)
print('\nFixes applied vs v1:')
print('  ✓ noise_gibberish — class-aware filter, class now has training data')
print('  ✓ Early stopping — no wasted epochs after plateau')
print('  ✓ ONNX — onnxscript installed, export works')
print('  ✓ Regex gate — inference_gate.py saved alongside model')
print('  ✓ num_workers=0 — no more DataLoader AssertionError noise')